# 銀行 融資ポートフォリオ分析 AI エージェント ハンズオン

このハンズオンでは、Snowflake の AI 機能を使って**融資商品マスタ・日次融資実行データ・残高データ・マクロ経済データ**の整備から **「自然言語で質問できる AI エージェント作成」** までを一気通貫で構築します。

### 今日つくるもの

銀行の融資ポートフォリオを横断的に分析できる AI エージェントです。  
完成すると、例えばこんな質問に自然言語で答えてくれます：

> 「2025年の月別融資実行金額ランキングを教えて。政策金利との相関も見たい」  
> 「現在、要管理・不良先になっている商品と、その実行推移を教えて」  
> 「金利が上昇した時期に実行が増える固定金利商品はどれ？」  
> 「個人ローン部と法人ローン部、部門別の実行金額を比較して」

---

### ハンズオンの流れ（所要時間：約80分 ＋ 応用25分）

| Step | やること | 学べる Snowflake 機能 | 所要時間 |
|:----:|----------|----------------------|:--------:|
| 0 | 事前準備・データ確認 | Snowflake Notebook | 5分 |
| 1 | AI で融資商品マスタを整備 | AI_CLASSIFY / AI_COMPLETE | 10分 |
| 2 | マクロ経済データを取り込む | Snowflake Marketplace | 10分 |
| 3 | 融資商品の意味検索を可能にする | Cortex Search Service | 10分 |
| 4 | データの意味を定義する | Semantic View（Cortex Analyst） | 20分 |
| 5 | AI エージェントを組み立てる | Cortex Agent | 15分 |
| 6 | 動かしてみよう！ | Cortex Agent UI | 5分 |
| 7 | 【オプション】エージェントの品質を評価する | Cortex Agent Evaluations | 15分 |
| 8 | AI 機能のコスト管理を設定する | Resource Budgets | 10分 |

## Step 0: 事前準備
使用するデータベース・スキーマ・ウェアハウスを設定します。

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.sql("SELECT CURRENT_USER()").collect()[0][0]
db_name = user.replace('.', '_').replace('-', '_') + '_FINANCIAL_AI_HANDSON'
session.sql(f"USE DATABASE {db_name}").collect()
session.sql("USE SCHEMA ANALYTICS").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()
print(f"Database: {db_name}")

今回操作するデータについて確認していきましょう。

まず融資商品マスタです。商品コード・商品名・部門情報が含まれています。  
今回は 4 部門（個人ローン部・法人ローン部・不動産ファイナンス部・中小企業支援部）の計 55 商品を扱います。

In [ ]:
%%sql -r dataframe_2
SELECT * FROM RAW.LOAN_MASTER;

次に日次融資実行データ。2022年から2026年2月までの実行実績です。

In [ ]:
%%sql -r dataframe_3
SELECT * FROM ANALYTICS.EXECUTION_DATA LIMIT 20;

最後に融資残高スナップショットデータです。2026-02-24 時点の残高状況（正常/要管理/不良）が確認できます。

In [ ]:
%%sql -r dataframe_4
SELECT * FROM ANALYTICS.LOAN_PORTFOLIO;

---
## Step 1: AI で融資商品マスタを整備する

現在の融資商品マスタ（`RAW.LOAN_MASTER`）には、商品コード・商品名・部門名しかありません。  
ここに Snowflake の **AI Functions** を使って、2つの情報を自動で追加します。

### 追加する情報

#### 1. 商品カテゴリの自動分類 — `AI_CLASSIFY`

商品名だけを見て、AI が自動的に商品カテゴリを判定します。  
あらかじめ定義した **5つのカテゴリ** に分類します（MECE＝漏れなくダブりなく）。

| カテゴリ | どんな商品？ |
|---------|-------------|
| 住宅・不動産ローン | 住宅購入・建設・リフォームなど居住用不動産を目的とした個人向けローン |
| 自動車・消費者ローン | 自動車購入・教育・医療・ブライダルなど個人の消費目的のローン |
| 事業・設備資金ローン | 法人の運転資金・設備投資・IT・環境対応を目的とした企業向け融資 |
| 不動産投資ローン | 収益物件・商業施設・オフィス・物流施設など不動産投資・開発を目的とした融資 |
| 創業・事業承継ローン | 新規創業・成長支援・事業承継・地方創生・農業など政策的意義を持つ中小企業支援融資 |

#### 2. 商品キャプションの自動生成 — `AI_COMPLETE`

商品名と部門名から、その融資商品の **特性・想定顧客・リスク・金利特性を説明する文章（約200文字）** を AI が自動で生成します。

> **ポイント**: `AI_CLASSIFY` はラベル＋説明文を渡すだけで分類してくれるので、従来のルールベースのロジックを書く必要がありません。

それでは実行しましょう。商品数にもよりますが、**約1〜2分** で完了します。

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE TABLE ANALYTICS.AI_LOAN_MASTER AS
SELECT
    LOAN_CODE,
    LOAN_NAME,
    DEPT_NAME,
    DEPT_CODE,
    -- ① AI_CLASSIFY 関数で商品カテゴリを分類（MECE）
    AI_CLASSIFY(
        LOAN_NAME,
        [
            {'label': '住宅・不動産ローン',    'description': '住宅購入・建設・リフォームなど居住用不動産を目的とした個人向けローン'},
            {'label': '自動車・消費者ローン',  'description': '自動車購入・教育・医療・ブライダルなど個人の消費目的のローン'},
            {'label': '事業・設備資金ローン',  'description': '法人の運転資金・設備投資・IT・環境対応を目的とした企業向け融資'},
            {'label': '不動産投資ローン',      'description': '収益物件・商業施設・オフィス・物流施設など不動産投資・開発を目的とした融資'},
            {'label': '創業・事業承継ローン',  'description': '新規創業・成長支援・事業承継・地方創生・農業など政策的意義を持つ中小企業支援融資'}
        ],
        {'task_description': '銀行の融資商品名から最も適切なカテゴリに分類してください'}
    ):labels[0]::VARCHAR AS CATEGORY,
    -- ② AI_COMPLETE 関数で商品キャプション生成
    TRIM(SNOWFLAKE.CORTEX.AI_COMPLETE(
        'llama3.1-405b',
        '以下の融資商品について、商品特性・想定顧客・リスク・金利特性を説明する文章（200文字程度）を日本語で1つ作成してください。\n商品名: ' || LOAN_NAME || '\n部門: ' || COALESCE(DEPT_NAME, '不明') || '\n説明文:'
    )) AS PRODUCT_CAPTION,
    CURRENT_TIMESTAMP() AS CREATED_AT
FROM RAW.LOAN_MASTER;

### 結果を見てみましょう

AI が生成した `CATEGORY`（商品カテゴリ）と `PRODUCT_CAPTION`（商品キャプション）を確認します。

In [ ]:
%%sql -r dataframe_6
SELECT * FROM ANALYTICS.AI_LOAN_MASTER;

### カテゴリ分類の精度を確認

5つのカテゴリにバランスよく分類されているか確認しましょう。

In [ ]:
%%sql -r dataframe_7
SELECT
    CATEGORY,
    COUNT(*) AS PRODUCT_COUNT
FROM ANALYTICS.AI_LOAN_MASTER
GROUP BY CATEGORY
ORDER BY PRODUCT_COUNT DESC;

---
## Step 2: マクロ経済データを取り込む（Snowflake Marketplace）

融資の需要・返済能力は金利環境やマクロ経済動向に大きく左右されます。  
住宅ローンは金利上昇で変動型から固定型へのシフトが起き、事業融資は景気後退で不良債権が増加します —  
こうした **金利・市場環境 × 融資実行の分析** を可能にするため、Snowflake Marketplace の **マクロ経済データ** を取り込みます。

### データソース

**FRED - Federal Reserve Economic Data Series Observations**（Fruit Stand 提供）のグローバルマクロ経済データを使用します。  
10万以上の FRED シリーズの完全な履歴データにアクセスでき、日本の金利・為替・株価データを取得できます。  
**30日間の無料トライアル** で全シリーズにフルアクセス可能です。

### 取り込み手順
1. Snowsight の左メニューから **Marketplace** をクリック
2. 上部の検索窓に `FRED Federal Reserve Economic Data` と入力
3. 表示された一覧から **FRED - Federal Reserve Economic Data Series Observations**（Fruit Stand 提供）をクリック
4. 右上の「**取得**」を押し、トライアルを開始する
5. データベース名を確認して「**取得**」を押す（デフォルト: `FRED__FEDERAL_RESERVE_ECONOMIC_DATA_SERIES_OBSERVATIONS`）

### 取得する指標

| 指標 | FRED コード | 説明 |
|------|------------|------|
| 政策金利 | `IRSTCI01JPM156N` | 日銀の無担保コール翌日物金利 |
| 長期金利 | `IRLTLT01JPM156N` | 日本10年国債利回り |
| ドル円レート | `DEXJPUS` | 円/ドル為替レート |
| 日経平均 | `NIKKEI225` | 日経225株価指数 |

### なぜテーブルを作るのか？

元データは SERIES_ID ごとに縦持ちフォーマットになっています。  
融資データと結合するには **日付ごとに各指標を1行に変換（ピボット）** する必要があります。

In [ ]:
%%sql -r dataframe_8
-- まず利用可能な変数を確認します
-- ※ FRED__FEDERAL_RESERVE_ECONOMIC_DATA_SERIES_OBSERVATIONS はMarketplaceで取得した際のデータベース名です
SELECT
    SERIES_ID,
    SERIES_NAME,
    UNITS
FROM FRED__FEDERAL_RESERVE_ECONOMIC_DATA_SERIES_OBSERVATIONS.FRED.FRED_SERIES_VW
WHERE SERIES_ID IN (
    'IRSTCI01JPM156N',
    'IRLTLT01JPM156N',
    'DEXJPUS',
    'NIKKEI225'
)
ORDER BY SERIES_ID;

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE TABLE ANALYTICS.MACRO_INDICATORS AS
WITH fred AS (
    SELECT
        DATE          AS MARKET_DATE,
        SERIES_ID,
        VALUE
    FROM FRED__FEDERAL_RESERVE_ECONOMIC_DATA_SERIES_OBSERVATIONS.FRED.FRED_SERIES_OBSERVATIONS_VW
    WHERE SERIES_ID IN (
        'IRSTCI01JPM156N',  -- 政策金利（BOJ翌日物）
        'IRLTLT01JPM156N',  -- 長期金利（10年国債利回り）
        'DEXJPUS',          -- ドル円レート
        'NIKKEI225'         -- 日経平均株価
    )
    AND DATE >= '2015-01-01'
)
SELECT
    MARKET_DATE,
    ROUND(MAX(CASE WHEN SERIES_ID = 'IRSTCI01JPM156N' THEN VALUE END), 3) AS POLICY_RATE,
    ROUND(MAX(CASE WHEN SERIES_ID = 'IRLTLT01JPM156N' THEN VALUE END), 3) AS LONG_RATE_10Y,
    ROUND(MAX(CASE WHEN SERIES_ID = 'DEXJPUS'         THEN VALUE END), 2) AS JPY_USD,
    ROUND(MAX(CASE WHEN SERIES_ID = 'NIKKEI225'        THEN VALUE END), 0) AS NIKKEI_AVG
FROM fred
GROUP BY MARKET_DATE
ORDER BY MARKET_DATE;

In [ ]:
%%sql -r dataframe_10
SELECT * FROM ANALYTICS.MACRO_INDICATORS
WHERE MARKET_DATE >= '2022-01-01'
LIMIT 20;

---
## Step 3: 融資商品の意味検索を可能にする（Cortex Search）

Step 1 で作った `AI_LOAN_MASTER` に対して、**Cortex Search Service** を作成します。

### Cortex Search とは？

テキストの **意味** を理解して検索するサービスです。  
キーワードの完全一致ではなく、「こういう意味のものを探して」という検索ができます。

例えば「金利上昇に強いローン」と聞くと、フラット35や固定特約商品がヒットします。

### 仕組み

商品名・部門名・カテゴリ・AI生成キャプションを **1つのテキストに結合** し、  
Snowflake の Embedding モデル（`snowflake-arctic-embed-l-v2.0`）でベクトル化して検索可能にします。

In [ ]:
%%sql -r dataframe_11
CREATE OR REPLACE CORTEX SEARCH SERVICE ANALYTICS.LOAN_SEARCH_SERVICE
    ON SEARCH_TEXT
    ATTRIBUTES LOAN_CODE, LOAN_NAME, DEPT_NAME, CATEGORY, PRODUCT_CAPTION
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT
        LOAN_CODE,
        LOAN_NAME,
        DEPT_NAME,
        DEPT_CODE,
        CATEGORY,
        PRODUCT_CAPTION,
        CONCAT(
            COALESCE(LOAN_NAME, ''), ' ',
            COALESCE(DEPT_NAME, ''), ' ',
            COALESCE(CATEGORY, ''), ' ',
            COALESCE(PRODUCT_CAPTION, '')
        ) AS SEARCH_TEXT
    FROM ANALYTICS.AI_LOAN_MASTER
);

---
## Step 4: データの意味を定義する（セマンティックビュー）

ここがこのハンズオンの **キモ** です。

Cortex Analyst に「住宅ローンの実行推移を教えて」と聞いたとき、どのテーブルの・どのカラムを・どう集計すればよいかを  
AI が理解できるように、**データの意味（セマンティクス）** を定義します。

### セマンティックビューで定義すること

| 定義項目 | 何を定義する？ | 具体例 |
|----------|---------------|--------|
| **テーブル** | 分析対象のテーブルとその説明 | EXECUTION_DATA, LOAN_PORTFOLIO, MACRO_INDICATORS |
| **リレーションシップ** | テーブル間の結合条件 | 実行日 = 市場観測日 |
| **ファクト** | 集計対象の数値カラム | 実行件数、実行総額、残高金額、政策金利 |
| **ディメンション** | 分析の切り口となる属性 | 商品名、部門、日付 |
| **メトリクス** | よく使う集計パターン | 融資実行総額＝SUM(実行総額) |
| **カスタムインストラクション** | SQL生成のガードレール | 数値の丸め、デフォルトフィルタ |

### 3つのテーブルの関係

```
                  LOAN_CODE                      EXECUTION_DATE = MARKET_DATE
EXECUTION_DATA ─────────────────▶ LOAN_PORTFOLIO  EXECUTION_DATA ──────────────▶ MACRO_INDICATORS
（日次融資実行）                    （残高スナップ）  （日次融資実行）                （マクロ経済）
```

> **Note**: 最新の Best Practices では、フロンティアモデル利用時にシノニムの大量定義は**非推奨**です。COMMENT に適切な説明を記載すれば、モデルは「融資」「実行」「loan execution」などの一般的な同義語を自動で理解します。

In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE SEMANTIC VIEW ANALYTICS.BANKING_ANALYST

    -- ========================================
    -- テーブル定義
    -- ========================================
    tables (
        EXECUTION_DATA AS ANALYTICS.EXECUTION_DATA
            PRIMARY KEY (LOAN_CODE, EXECUTION_DATE)
            WITH SYNONYMS = ('実行データ', '取引データ', '融資実行')
            COMMENT = '日次融資実行データ。2022-01-01〜2026-02-24の商品別日次実行件数・基準融資額・実行総額を含む。',

        PORTFOLIO AS ANALYTICS.LOAN_PORTFOLIO
            PRIMARY KEY (LOAN_CODE)
            WITH SYNONYMS = ('残高', 'ポートフォリオ', '与信')
            COMMENT = '融資残高スナップショットデータ（2026-02-24時点）。商品別の残高件数・残高金額・与信ステータス（正常/要管理/不良）を含む。',

        MACRO AS ANALYTICS.MACRO_INDICATORS
            PRIMARY KEY (MARKET_DATE)
            COMMENT = '日次マクロ経済指標（2015年〜）。政策金利・10年国債利回り・ドル円レート・日経平均株価。'
    )

    -- ========================================
    -- リレーションシップ定義
    -- ========================================
    relationships (
        EXEC_PORTFOLIO AS EXECUTION_DATA(LOAN_CODE)      REFERENCES PORTFOLIO(LOAN_CODE),
        EXEC_MACRO     AS EXECUTION_DATA(EXECUTION_DATE) REFERENCES MACRO(MARKET_DATE)
    )

    -- ========================================
    -- ファクト定義（数値データ）
    -- ========================================
    facts (
        -- 融資実行ファクト
        EXECUTION_DATA.exec_count_fact   AS EXECUTION_COUNT
            COMMENT = '融資実行件数（件）',
        EXECUTION_DATA.base_amount_fact  AS BASE_AMOUNT
            COMMENT = '基準融資額（円）',
        EXECUTION_DATA.total_amount_fact AS TOTAL_AMOUNT
            COMMENT = '融資実行総額（円）',

        -- 残高ファクト
        PORTFOLIO.balance_count_fact  AS BALANCE_COUNT
            COMMENT = '融資残高件数（件）',
        PORTFOLIO.base_amount_fact    AS BASE_AMOUNT
            COMMENT = '基準融資額（円）',
        PORTFOLIO.balance_amount_fact AS BALANCE_AMOUNT
            COMMENT = '融資残高総額（円）',

        -- マクロ経済ファクト
        MACRO.policy_rate_fact  AS POLICY_RATE
            COMMENT = '政策金利（%）：日本銀行の無担保コール翌日物誘導目標金利',
        MACRO.long_rate_fact    AS LONG_RATE_10Y
            COMMENT = '長期金利・10年国債利回り（%）',
        MACRO.jpy_usd_fact      AS JPY_USD
            COMMENT = 'ドル円為替レート（円/ドル）',
        MACRO.nikkei_avg_fact   AS NIKKEI_AVG
            COMMENT = '日経平均株価（円）'
    )

    -- ========================================
    -- ディメンション定義（属性データ）
    -- ========================================
    dimensions (
        -- ========== EXECUTION_DATA ディメンション ==========
        EXECUTION_DATA.loan_code       AS LOAN_CODE
            COMMENT = '融資商品コード',
        EXECUTION_DATA.loan_name       AS LOAN_NAME
            COMMENT = '融資商品名',
        EXECUTION_DATA.dept_name       AS DEPT_NAME
            COMMENT = '部門名（個人ローン部・法人ローン部・不動産ファイナンス部・中小企業支援部）',
        EXECUTION_DATA.execution_date  AS EXECUTION_DATE
            COMMENT = '融資実行日（DATE型、2022-01-01〜2026-02-24）',
        EXECUTION_DATA.exec_year       AS YEAR(EXECUTION_DATE)
            COMMENT = '融資実行年（2022〜2026）',
        EXECUTION_DATA.exec_month      AS MONTH(EXECUTION_DATE)
            COMMENT = '融資実行月（1〜12）',

        -- ========== PORTFOLIO ディメンション ==========
        PORTFOLIO.loan_code     AS LOAN_CODE
            COMMENT = '融資商品コード',
        PORTFOLIO.loan_name     AS LOAN_NAME
            COMMENT = '融資商品名',
        PORTFOLIO.dept_name     AS DEPT_NAME
            COMMENT = '部門名',
        PORTFOLIO.snapshot_date AS SNAPSHOT_DATE
            COMMENT = '残高スナップショット日（2026-02-24）',
        PORTFOLIO.credit_status AS CREDIT_STATUS
            COMMENT = '与信ステータス（正常 / 要管理 / 不良）',

        -- ========== MACRO ディメンション ==========
        MACRO.market_date   AS MARKET_DATE
            COMMENT = 'マクロ経済データの観測日（DATE型、2015年〜）。EXECUTION_DATEとリレーションで紐づく。',
        MACRO.market_year   AS YEAR(MARKET_DATE)
            COMMENT = 'マクロ経済データの観測年',
        MACRO.market_month  AS MONTH(MARKET_DATE)
            COMMENT = 'マクロ経済データの観測月（1〜12）'
    )

    -- ========================================
    -- メトリクス定義（集計関数）
    -- ========================================
    metrics (
        -- 融資実行メトリクス
        EXECUTION_DATA.total_exec_amount  AS SUM(EXECUTION_DATA.total_amount_fact)
            COMMENT = '融資実行総額（円）',
        EXECUTION_DATA.total_exec_count   AS SUM(EXECUTION_DATA.exec_count_fact)
            COMMENT = '融資実行件数合計（件）',
        EXECUTION_DATA.avg_base_amount    AS AVG(EXECUTION_DATA.base_amount_fact)
            COMMENT = '平均基準融資額（円）',
        EXECUTION_DATA.avg_daily_exec     AS AVG(EXECUTION_DATA.total_amount_fact)
            COMMENT = '日次平均融資実行額（円）',

        -- 残高メトリクス
        PORTFOLIO.total_balance_count   AS SUM(PORTFOLIO.balance_count_fact)
            COMMENT = '融資残高件数合計（件）',
        PORTFOLIO.total_balance_amount  AS SUM(PORTFOLIO.balance_amount_fact)
            COMMENT = '融資残高総額（円）',

        -- マクロ経済メトリクス
        MACRO.period_avg_policy_rate  AS AVG(MACRO.policy_rate_fact)
            COMMENT = '期間内の平均政策金利（%）',
        MACRO.period_avg_long_rate    AS AVG(MACRO.long_rate_fact)
            COMMENT = '期間内の平均長期金利（%）',
        MACRO.period_avg_jpy_usd      AS AVG(MACRO.jpy_usd_fact)
            COMMENT = '期間内の平均ドル円レート（円/ドル）',
        MACRO.period_avg_nikkei       AS AVG(MACRO.nikkei_avg_fact)
            COMMENT = '期間内の平均日経平均株価（円）'
    )

    COMMENT = '銀行の融資実行・残高・マクロ経済指標の総合分析セマンティックビュー。個人ローン部・法人ローン部・不動産ファイナンス部・中小企業支援部の55商品の日次融資実行データ（2022年〜2026年2月）、残高スナップショット（2026-02-24時点）、およびマクロ経済指標（政策金利・長期金利・為替・株価）を統合。金利環境と融資実行の相関分析が可能。'

    -- ========================================
    -- カスタムインストラクション
    -- ========================================
    AI_SQL_GENERATION '数値カラムはROUND(x, 2)で小数点以下2桁に丸めてください。融資金額は円単位で表示してください。日付フィルタが指定されない場合は直近1年間をデフォルトにしてください。ランキングにはQUALIFY + ROW_NUMBERを使用してください。'
    AI_QUESTION_CATEGORIZATION '融資実行・残高・与信・マクロ経済に関する質問のみ回答してください。個人情報・融資先の機密情報・与信判断の依頼には「対応範囲外です。融資実行・残高・マクロ経済データに関する質問をお願いします。」と回答してください。';

### セマンティックビューの確認

正しく定義できたか確認しましょう。テーブル・ファクト・ディメンション・メトリクスの一覧が表示されます。

In [ ]:
%%sql -r dataframe_13
DESCRIBE SEMANTIC VIEW BANKING_ANALYST;

---
## Step 5: AI エージェントを組み立てる（Cortex Agent）

いよいよエージェントを組み立てます。  
ここまで作ってきた **Cortex Search** と **Cortex Analyst** を組み合わせて、  
1つの **AI エージェント** にまとめます。

### エージェントの仕組み

```
ユーザーの質問
     │
     ▼
┌─────────────────────────┐
│    Banking Agent       │
│  （オーケストレーター）   │
│                        │
│  質問の意図を判断して      │
│  適切なツールを選択        │
└────┬──────────┬─────────┘
     │          │
     ▼          ▼
┌─────────┐ ┌──────────────┐
│ Cortex  │ │   Cortex     │
│ Search  │ │  Analyst     │
│         │ │              │
│商品情報  │ │実行・残高・   │
│の意味検索│ │金利の数値分析  │
└─────────┘ └──────────────┘
```

### ツールの使い分け

| 質問の種類 | 使われるツール | 例 |
|-----------|---------------|----|
| 商品の特徴・情報を知りたい | **Cortex Search** | 「金利上昇に強いローン商品を教えて」 |
| 数値の集計・分析をしたい | **Cortex Analyst** | 「2025年の月別実行金額ランキングは？」 |
| 両方が必要 | **Search → Analyst** | 「要管理先商品の過去の実行推移は？」 |
| 外部情報が必要 | **Web Search** | 「2026年の金利見通しと住宅ローンへの影響は？」 |

In [ ]:
import json

db_name_df = session.sql("SELECT $DB_NAME AS DB_NAME").collect()
db_name = db_name_df[0]['DB_NAME']

spec = {
  "models": {"orchestration": "claude-4-sonnet"},
  "orchestration": {"budget": {"seconds": 60, "tokens": 32000}},
  "instructions": {
    "response": "あなたは銀行の融資アナリストです。融資実行データ（実行件数・基準融資額・実行総額）、融資残高データ（残高件数・残高金額・与信ステータス）、マクロ経済指標（政策金利・長期金利・ドル円・日経平均）を活用して回答してください。数値データは適切にフォーマットして表示してください（金額は円単位、件数は件単位）。日本語で丁寧に回答してください。可能であれば視覚化を提供してください。金利環境と融資実行の関係（政策金利上昇と変動型・固定型ローンの需要シフト、景気変動と不良債権の関係など）にも注意して分析してください。",
    "orchestration": "融資商品の詳細情報（商品名・部門・カテゴリ・商品特性など）を検索する場合はcortex searchを使用し、融資実行・残高・マクロ経済の数値分析にはcortex analystを使用してください。複合的な質問の場合は、まずcortex searchで商品を特定し、cortex analystに渡して分析してください。",
    "sample_questions": [
      {"question": "2025年の月別融資実行金額ランキングを商品別で可視化してください。政策金利の推移との相関も見たいです。"},
      {"question": "現在、要管理・不良先になっている融資商品を教えてください。それぞれの過去の実行推移も見せてください。"},
      {"question": "2024年の政策金利引き上げ後、変動金利商品と固定金利商品の実行件数はどう変化しましたか？"},
      {"question": "個人ローン部と法人ローン部、部門別の年間融資実行金額を比較してください。"},
      {"question": "融資実行総額が最も高い商品トップ10を教えてください。"}
    ]
  },
  "tools": [
    {
      "tool_spec": {
        "type": "cortex_analyst_text_to_sql",
        "name": "BANKING_ANALYST",
        "description": "銀行の日次融資実行データ、融資残高スナップショット、マクロ経済指標をクエリして分析します。商品別・部門別・日付別の実行件数・実行金額・残高金額の集計、与信ステータスの確認、政策金利・長期金利・為替・日経平均と融資実行の相関分析に使用します。"
      }
    },
    {
      "tool_spec": {
        "type": "cortex_search",
        "name": "LOAN_SEARCH",
        "description": "融資商品マスタから商品情報をセマンティック検索します。商品名・部門名・AIが分類した商品カテゴリ（住宅・不動産ローン、自動車・消費者ローン、事業・設備資金ローン、不動産投資ローン、創業・事業承継ローン）、AIが生成した商品キャプションなどのテキスト検索に使用します。"
      }
    },
    {
      "tool_spec": {
        "type": "web_search",
        "name": "WEB_SEARCH",
        "description": "インターネットから最新のニュースや外部情報を検索します。金融政策の動向・日銀の政策変更・景気見通し・不動産市況・業界規制など外部情報取得に使用します。"
      }
    }
  ],
  "tool_resources": {
    "BANKING_ANALYST": {
      "semantic_view": f"{db_name}.ANALYTICS.BANKING_ANALYST",
      "execution_environment": {"type": "warehouse", "warehouse": "COMPUTE_WH"}
    },
    "LOAN_SEARCH": {
      "name": f"{db_name}.ANALYTICS.LOAN_SEARCH_SERVICE",
      "max_results": 10
    },
    "WEB_SEARCH": {"max_results": 10}
  }
}

spec_json = json.dumps(spec, ensure_ascii=False, indent=2)

ddl = f"""CREATE OR REPLACE AGENT {db_name}.ANALYTICS.BANKING_AGENT
    COMMENT = '銀行の融資実行・残高・マクロ経済指標の分析と融資商品検索を行うエージェント'
FROM SPECIFICATION $${spec_json}$$"""

session.sql(ddl).collect()
print(f"Agent {db_name}.ANALYTICS.BANKING_AGENT を作成しました。")

**補足**

こちらは通常、SQL で作成することが可能です。今回のハンズオンで実行したものは一見 Python ですが、SQL の組み立てに Python を利用しているだけで実態としては SQL の実行処理になっています。
今回のハンズオンでは、同一 Snowflake アカウント上で複数ユーザーが同時にハンズオンを行うことを想定しているため、それを考慮した実装になるように Python による実装を行っています。
main branch の Notebook を確認いただくと、SQL の形式でご確認いただくことができるかと思います。

### エージェントの確認

In [ ]:
%%sql -r dataframe_15
SHOW AGENTS IN SCHEMA ANALYTICS;

---
## Step 6: 動かしてみよう！

エージェントが完成しました！Snowsight の **Cortex Agent UI** から以下のような質問を試してみましょう。

### Web Search を有効にする
なお、以下を質問する前に Web Search を有効にする必要があります。
- AI&ML > エージェント > 右上の **Settings**
- "Web Search" をクリックして **ON** にする

### 試してみたい質問例

**融資実行分析**
- 「2025年の月別融資実行金額ランキングを商品別で可視化してください。」
- 「融資実行総額が最も高い商品トップ10を教えてください。」
- 「2024年と2025年で実行金額が大きく増加した商品を教えてください。」

**金利環境 × 融資分析**
- 「2024年3月の政策金利引き上げ前後で、変動金利商品と固定金利商品の実行件数はどう変化しましたか？」
- 「長期金利と住宅ローン（フラット35）の実行件数の相関を分析してください。」
- 「ドル円為替レートと輸出入決済ローンの実行金額の関係を教えてください。」

**与信管理**
- 「現在、要管理・不良先になっている融資商品を教えてください。それぞれの過去の実行推移も見せてください。」
- 「与信ステータスが要管理の商品はどれですか？残高金額と合わせて確認したいです。」

**商品・部門分析**
- 「個人ローン部と法人ローン部、部門別の年間融資実行金額を比較してください。」
- 「IT・DX投資ローンと環境・脱炭素設備ローンの実行推移を教えてください。成長トレンドも見せて。」

**Web検索**
- 「2026年の日銀の金融政策見通しと、住宅ローン市場への影響を教えてください。」
- 「最近の中小企業向け融資の業界トレンドを教えてください。」

### Step 6-1: Snowflake Intelligence でも動かしてみよう！

1. エージェントの「概要」欄左上にある、「エージェントを追加」ボタンを押します。
2. 次に、右上の「Snowflake Intelligence でのプレビュー」ボタンを押します
3. 認証が求められた場合は、デモアカウントのユーザー名とパスワードを入力してログインしてください
4. エージェントに「BANKING_AGENT」が選択されていることを確認して上記と同じ質問をしてみましょう。ぜひ自分なりの質問もしてみてください！


---
## 【オプション】Step 7: エージェントの品質を評価する（Cortex Agent Evaluations）
**注意** こちらの手順はトライアルアカウントでは実行できません。実行されたい場合は、有償でのご契約があるアカウントにて実行する必要があります。

AI エージェントを構築することは第一歩に過ぎません。本番環境で信頼性の高いアプリケーションを提供するためには、エージェントのパフォーマンスを **定量的に評価** することが不可欠です。

**Cortex Agent Evaluations** は、Cortex Agent のパフォーマンスを複数の観点から測定するためのネイティブ機能です。SQL だけで評価の実行から結果取得まで完結できます。

### 組み込みメトリクス

| メトリクス | 説明 | 特徴 |
|-----------|------|------|
| **Answer Correctness** | エージェントの回答が期待される回答（Ground Truth）とどれだけ一致するか | Ground Truth が必要 |
| **Logical Consistency** | エージェントの指示、計画、ツール呼び出し全体の一貫性 | Reference-free（Ground Truth 不要） |

### 評価の流れ

```
評価データセット作成 → データセット登録 → YAML設定 → 評価実行 → 結果確認
```

> **参考**: [Snowflake Cortex Agent Evaluationsがリリースされました](https://zenn.dev/snowflakejp/articles/258c73c6ec4054)

### 評価データセットの作成

まず、エージェントに対する質問と期待される回答（Ground Truth）のペアを作成します。

> **ポイント**: Ground Truth は LLM ジャッジのプロンプトに含まれるため、「期待される回答の自然言語での記述」が有効です。  
> 具体的なデータ値を含めると Answer Correctness が正確に測定されます。

In [ ]:
%%sql -r dataframe_16
-- 評価データセットテーブルの作成
CREATE OR REPLACE TABLE ANALYTICS.AGENT_EVAL_DATASET (
    input_query  VARCHAR,
    ground_truth VARIANT
);

-- 評価データの投入
INSERT INTO ANALYTICS.AGENT_EVAL_DATASET
SELECT '2025年の月別合計融資実行金額を教えてください' AS input_query,
       PARSE_JSON('{"ground_truth_output": "EXECUTION_DATAテーブルからEXECUTION_DATEが2025年のレコードを対象に、月別にTOTAL_AMOUNTを集計した結果を回答する。金額は円単位で表示する。"}') AS ground_truth
UNION ALL
SELECT '現在、与信ステータスが不良の融資商品はどれですか？',
       PARSE_JSON('{"ground_truth_output": "LOAN_PORTFOLIOテーブルからCREDIT_STATUSが不良のレコードを検索し、商品名・部門名・残高件数・残高金額を含めて回答する。"}')
UNION ALL
SELECT '融資実行総額が最も高い商品トップ5を教えてください',
       PARSE_JSON('{"ground_truth_output": "EXECUTION_DATAテーブルから商品別にTOTAL_AMOUNTを集計し、上位5商品をランキング形式で回答する。"}')
UNION ALL
SELECT '政策金利と住宅ローン変動型の実行件数の関係を分析してください',
       PARSE_JSON('{"ground_truth_output": "EXECUTION_DATAとMACRO_INDICATORSをEXECUTION_DATE=MARKET_DATEで結合し、住宅ローン変動型商品の実行件数と政策金利の相関関係を分析した結果を提示する。2024年の利上げ前後での変化に注目する。"}')
UNION ALL
SELECT '個人ローン部と法人ローン部の年間融資実行金額を比較してください',
       PARSE_JSON('{"ground_truth_output": "EXECUTION_DATAテーブルからDEPT_NAMEが個人ローン部または法人ローン部のレコードを年別・部門別にTOTAL_AMOUNTを集計し、比較結果を回答する。"}');

### 評価設定 YAML の準備

評価を実行するには、YAML 設定ファイルをステージにアップロードする必要があります。  
設定ファイルは `eval_config.yaml` という名前でワークスペース上にあります。
具体的な内容は以下となっています。

```yaml
# データセットの登録
dataset:
  dataset_type: "CORTEX AGENT"
  table_name: "FINANCIAL_AI_HANDSON.ANALYTICS.AGENT_EVAL_DATASET"
  dataset_name: "FINANCIAL_EVAL_DS"
  column_mapping:
    query_text: "INPUT_QUERY"
    ground_truth: "GROUND_TRUTH"

# エージェント評価の設定
evaluation:
  agent_params:
    agent_name: "FINANCIAL_AI_HANDSON.ANALYTICS.BANKING_AGENT"
    agent_type: "CORTEX AGENT"
  run_params:
    label: "banking_baseline"
    description: "Banking agent baseline evaluation"
  source_metadata:
    type: "dataset"
    dataset_name: "FINANCIAL_EVAL_DS"

# 評価メトリクス（トップレベルキー）
metrics:
  - "answer_correctness"
  - "logical_consistency"
```

Snow CLI でアップロードする場合は以下のコマンドを利用いただけます:
```bash
snow stage copy eval_config.yaml @FINANCIAL_AI_HANDSON.ANALYTICS.EVAL_CONFIG_STAGE --overwrite
```

> **重要**: `source_metadata.type` は小文字で指定（`"dataset"`）。  
> ステージは次のセルで FILE FORMAT 付きで作成します。FILE FORMAT がないと YAML のパースに失敗します。

In [ ]:
%%sql -r dataframe_17
# eval_config.yaml を動的に生成してステージにアップロード

# セッション変数 DB_NAME を取得
db_name_df = session.sql("SELECT $DB_NAME AS DB_NAME").collect()
db_name = db_name_df[0]['DB_NAME']

# FILE FORMAT とステージの作成
session.sql(f"""
CREATE OR REPLACE FILE FORMAT {db_name}.ANALYTICS.YAML_FILE_FORMAT
    TYPE = 'CSV'
    FIELD_DELIMITER = NONE
    RECORD_DELIMITER = '\n'
    SKIP_HEADER = 0
    FIELD_OPTIONALLY_ENCLOSED_BY = NONE
    ESCAPE_UNENCLOSED_FIELD = NONE
""").collect()

session.sql(f"""
CREATE OR REPLACE STAGE {db_name}.ANALYTICS.EVAL_CONFIG_STAGE
    FILE_FORMAT = {db_name}.ANALYTICS.YAML_FILE_FORMAT
""").collect()

# eval_config.yaml の内容を動的に生成
eval_config_yaml = f"""# データセットの登録
dataset:
  dataset_type: "CORTEX AGENT"
  table_name: "{db_name}.ANALYTICS.AGENT_EVAL_DATASET"
  dataset_name: "FINANCIAL_EVAL_DS"
  column_mapping:
    query_text: "INPUT_QUERY"
    ground_truth: "GROUND_TRUTH"

# エージェント評価の設定
evaluation:
  agent_params:
    agent_name: "{db_name}.ANALYTICS.BANKING_AGENT"
    agent_type: "CORTEX AGENT"
  run_params:
    label: "banking_baseline"
    description: "Banking agent baseline evaluation"
  source_metadata:
    type: "dataset"
    dataset_name: "FINANCIAL_EVAL_DS"

# 評価メトリクス（トップレベルキー）
metrics:
  - "answer_correctness"
  - "logical_consistency"
"""

# 一時ファイルに書き出してステージに PUT
import tempfile, os
with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False, prefix='eval_config_') as f:
    f.write(eval_config_yaml)
    tmp_path = f.name

session.sql(f"PUT 'file://{tmp_path}' @{db_name}.ANALYTICS.EVAL_CONFIG_STAGE/ AUTO_COMPRESS=FALSE OVERWRITE=TRUE").collect()
os.unlink(tmp_path)

print(f"eval_config.yaml を @{db_name}.ANALYTICS.EVAL_CONFIG_STAGE/ にアップロードしました。")

In [ ]:
%%sql -r dataframe_24
# 評価の開始
db_name_df = session.sql("SELECT $DB_NAME AS DB_NAME").collect()
db_name = db_name_df[0]['DB_NAME']

result = session.sql(f"""
CALL EXECUTE_AI_EVALUATION(
    'START',
    OBJECT_CONSTRUCT('run_name', 'banking_eval_01'),
    '@{db_name}.ANALYTICS.EVAL_CONFIG_STAGE/eval_config.yaml'
)
""").collect()
print(result)

In [ ]:
%%sql -r dataframe_18
# ステータス確認（COMPLETED になるまで繰り返し実行）
db_name_df = session.sql("SELECT $DB_NAME AS DB_NAME").collect()
db_name = db_name_df[0]['DB_NAME']

result = session.sql(f"""
CALL EXECUTE_AI_EVALUATION(
    'STATUS',
    OBJECT_CONSTRUCT('run_name', 'banking_eval_01'),
    '@{db_name}.ANALYTICS.EVAL_CONFIG_STAGE/eval_config.yaml'
)
""").collect()
print(result)

### 結果の確認

評価が完了したら、以下の方法で結果を確認できます。

**Snowsight UI での確認:**
1. **AI \& ML** > **Agents** を選択
2. 対象のエージェント（`BANKING_AGENT`）をクリック
3. **Evaluations** タブを選択

**確認できる指標:**
- **AC（Answer Correctness）**: 回答の正確性スコア（0〜1、高いほど良い）
- **LC（Logical Consistency）**: 論理的一貫性スコア（0〜1、高いほど良い）

以下の SQL でも結果を取得できます。

In [ ]:
%%sql -r dataframe_19
-- 評価結果の詳細を取得
SELECT * FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_RESULTS('banking_eval_01'));

---
## Step 8: AI 機能のコスト管理を設定する（Resource Budgets）

AI 機能を活用するほど、コンピュートクレジットの消費が増えます。  
**Resource Budgets** を使うことで、AI 機能のコストを監視・制御できます。

### Resource Budgets とは？

Snowflake オブジェクト（ウェアハウス、タスク、サービスなど）に対して  
**予算上限を設定し、超過時にアラートを受け取れる** コスト管理機能です。

### 設定の流れ

```
Budget 作成 → オブジェクト紐付け → 消費状況確認
```

### タグの作成とエージェントへの適用

まず、AI 機能をグループ化するためのタグを作成し、エージェントに適用します。

In [ ]:
%%sql -r dataframe_20
-- コスト管理用タグの作成
CREATE TAG IF NOT EXISTS ANALYTICS.FINANCIAL_AI_TAG
    COMMENT = '金融 AI 機能のコスト追跡用タグ';

-- エージェントにタグを適用
ALTER AGENT ANALYTICS.BANKING_AGENT
    SET TAG ANALYTICS.FINANCIAL_AI_TAG = 'financial-ai-handson';

### Budget の作成と設定

In [ ]:
# Resource Budget の作成（DB名を動的に埋め込み）
db_name_df = session.sql("SELECT $DB_NAME AS DB_NAME").collect()
db_name = db_name_df[0]['DB_NAME']

# SNOWFLAKE アプリケーションへの権限付与（Budget 管理に必要）
session.sql(f"GRANT USAGE ON DATABASE {db_name} TO APPLICATION SNOWFLAKE").collect()
session.sql(f"GRANT USAGE ON SCHEMA {db_name}.ANALYTICS TO APPLICATION SNOWFLAKE").collect()

# Resource Budget の作成
session.sql(f"""
CREATE OR REPLACE SNOWFLAKE.CORE.BUDGET {db_name}.ANALYTICS.FINANCIAL_AI_BUDGET()
    COMMENT = '金融 AI ハンズオン用 Resource Budget（月間100クレジット上限）'
""").collect()

# 月間100クレジット上限の設定
session.sql(f"CALL {db_name}.ANALYTICS.FINANCIAL_AI_BUDGET!SET_SPENDING_LIMIT(100)").collect()

print(f"Resource Budget {db_name}.ANALYTICS.FINANCIAL_AI_BUDGET を作成し、月間100クレジット上限を設定しました。")

### 使用量の確認

Budget に紐づくリソースのクレジット使用量を確認できます。  
データの反映には最大6.5時間（TIER_1H に設定した場合は最大1.5時間）かかる場合があります。

In [ ]:
%%sql -r dataframe_22
-- バジェット一覧の確認
SHOW SNOWFLAKE.CORE.BUDGET IN SCHEMA ANALYTICS;

In [ ]:
# バジェットの消費状況確認（過去7日間）
db_name_df = session.sql("SELECT $DB_NAME AS DB_NAME").collect()
db_name = db_name_df[0]['DB_NAME']

result = session.sql(f"""
CALL {db_name}.ANALYTICS.FINANCIAL_AI_BUDGET!GET_SPENDING_HISTORY(
    TIME_LOWER_BOUND => DATEADD('days', -7, CURRENT_TIMESTAMP()),
    TIME_UPPER_BOUND => CURRENT_TIMESTAMP()
)
""").collect()
print(result)

### 設定されたバジェットの確認

以下の方法で設定済みのバジェットを確認できます。

**Snowsight UI での確認:**
1. 左メニューの **管理（Admin）** をクリック
2. **コスト管理** を選択
3. **Resource Budgets** をクリック
4. 作成した `FINANCIAL_AI_BUDGET` を選択して消費状況を確認

---

## おつかれさまでした！

これで **銀行 融資ポートフォリオ分析 AI エージェント ハンズオン** のすべてのステップが完了です。

### 今日学んだこと

| Step | 機能 | できるようになったこと |
|:----:|------|----------------------|
| 0 | Snowflake Notebook | データの確認・SQL の実行 |
| 1 | AI_CLASSIFY / AI_COMPLETE | 融資商品カテゴリの自動分類・キャプション生成 |
| 2 | Snowflake Marketplace | 外部データ（マクロ経済指標）の取得・変換 |
| 3 | Cortex Search Service | 融資商品のテキスト意味検索サービス構築 |
| 4 | Semantic View | データの意味定義（Cortex Analyst 連携） |
| 5 | Cortex Agent | 複数ツールを統合した AI エージェント構築 |
| 6 | Cortex Agent UI | 自然言語でのデータ分析 |
| 7 | Agent Evaluations | エージェントの品質定量評価 |
| 8 | Resource Budgets | AI コストの監視・管理 |

### 次のステップ

- エージェントに質問を追加してみましょう
- Semantic View のメトリクスを追加してより深い分析を実現しましょう
- Agent Evaluations で定期的に品質を測定しましょう